# Docker Fundamentals — Docker Compose Assignment

**Task:** Use Docker Compose to define and run multi-container applications.

We'll build a realistic 3-container stack:
- **web** — Flask API
- **db** — PostgreSQL
- **cache** — Redis

all wired together with a single `docker-compose.yml`.

## 1. Application Code

In [1]:
import os
os.makedirs('project/web', exist_ok=True)

with open('project/web/app.py', 'w') as f:
    f.write('''import os
import redis
import psycopg2
from flask import Flask, jsonify

app = Flask(__name__)
r = redis.Redis(host="cache", port=6379, decode_responses=True)

def get_db_conn():
    return psycopg2.connect(
        host="db", dbname=os.environ["POSTGRES_DB"],
        user=os.environ["POSTGRES_USER"], password=os.environ["POSTGRES_PASSWORD"]
    )

@app.route("/")
def home():
    visits = r.incr("visits")   # demonstrates the cache container
    return jsonify({"message": "Hello from Compose!", "visits": visits})

@app.route("/db-check")
def db_check():
    conn = get_db_conn()
    cur = conn.cursor()
    cur.execute("SELECT 1;")
    result = cur.fetchone()
    cur.close(); conn.close()
    return jsonify({"db_ok": result[0] == 1})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
''')

with open('project/web/requirements.txt', 'w') as f:
    f.write("flask\nredis\npsycopg2-binary\n")

with open('project/web/Dockerfile', 'w') as f:
    f.write('''FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 5000
CMD ["python", "app.py"]
''')
print("Created project/web/{app.py, requirements.txt, Dockerfile}")


Created project/web/{app.py, requirements.txt, Dockerfile}


## 2. `docker-compose.yml`

In [2]:
with open('project/docker-compose.yml', 'w') as f:
    f.write('''version: "3.9"

services:
  web:
    build: ./web
    ports:
      - "5000:5000"
    environment:
      POSTGRES_DB: appdb
      POSTGRES_USER: appuser
      POSTGRES_PASSWORD: apppass
    depends_on:
      db:
        condition: service_healthy
      cache:
        condition: service_started
    networks: [appnet]

  db:
    image: postgres:16-alpine
    environment:
      POSTGRES_DB: appdb
      POSTGRES_USER: appuser
      POSTGRES_PASSWORD: apppass
    volumes:
      - db_data:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U appuser -d appdb"]
      interval: 5s
      timeout: 3s
      retries: 5
    networks: [appnet]

  cache:
    image: redis:7-alpine
    networks: [appnet]

volumes:
  db_data:

networks:
  appnet:
    driver: bridge
''')
print("Created project/docker-compose.yml")
print("Run:   docker compose up --build")
print("Check: curl http://localhost:5000/          -> visit counter increments (Redis)")
print("       curl http://localhost:5000/db-check   -> {'db_ok': true}  (Postgres)")


Created project/docker-compose.yml
Run:   docker compose up --build
Check: curl http://localhost:5000/          -> visit counter increments (Redis)
       curl http://localhost:5000/db-check   -> {'db_ok': true}  (Postgres)


## 3. Useful Compose Commands
```bash
docker compose up --build -d      # build & start all services in the background
docker compose ps                 # see running containers
docker compose logs -f web        # tail logs for one service
docker compose exec db psql -U appuser -d appdb   # shell into the DB container
docker compose down -v            # stop and remove containers + volumes
```